Note: Before running this notebook update the table so you do not repull the data.

In [ ]:
# You will need to set these variables for your workspace.
year =

DB_VOLUME = "/Volumes/sae1_prod_cimmyt_fg_catalog_7474645209598392/tier1_raw/data"
PROJECT_DIR = f"{DB_VOLUME}/hiphen"
TABLE_NAME = "sae1_prod_cimmyt_fg_catalog_7474645209598392.tier1_raw.hipen_missions"

SITE_ID = "cgiar-cloverfield-api"
CLIENT_ID = "cgiar-cloverfield-api"
DEFAULT_BASE_URL = "https://api.hiphen-cloverfield.com"

In [ ]:
%pip install "typing_extensions>=4.14.0" git+https://github.com/AgDMALabs-Public/ag-vision-dataops.git


In [ ]:
dbutils.library.restartPython()

In [ ]:
import os

import pandas as pd
from ag_vision.external_connections.hiphen import utils as hu

In [ ]:
client_secret = dbutils.secrets.get(scope="hiphen",
                                    key="hiphen-api-key")

In [ ]:
mission_df = spark.read.table(TABLE_NAME).toPandas()

In [ ]:
mission_df.sample(10)

In [ ]:
token_mgr = hu.TokenManager(base_url=DEFAULT_BASE_URL,
                            client_id=CLIENT_ID,
                            client_secret=client_secret)

token_mgr.get_headers()

In [ ]:
for idx, row in mission_df.iterrows():
    if row['plot_img_count'] > 0:
        continue
    elif row['crop'].lower() == 'unknown':
        # We will not save data until the crop has been set in cloverfield.
        continue
    else:
        hi = hu.HiphenData(token_mgr=token_mgr,
                           year=int(row['year']),
                           country=row['country'],
                           crop=row['crop'].lower(),
                           season=row['season'],
                           mission_name='hiphen flights',
                           camera='rgb',
                           site_name=row['contract'],
                           trial_name='hiphen_data',
                           field_name=row['location'],
                           location_name=row['location'],
                           project_dir=PROJECT_DIR)

        hi.set_crop_season()
        hi.set_mission_dir()
        hi.list_contracts()
        hi.set_contract_id(index=1) # this will be set with the table
        hi.list_sites()
        hi.set_site_id(index=row['site_idx'])
        hi.get_site_info()
        hi.set_flight_date(flight_date=row['flight_date'])
        hi.get_plot_image_list()
        hi.get_ortho_img_path()
        hi.get_hiphen_plot_metrics()
        hi.generate_results_dataframe()
        hi.generate_results_geojson()
        hi.save_tabular_results()
        hi.save_geojson_results()
        hi.download_plot_images()
        hi.download_ortho_images()